In [1]:
# Install libraries
!pip install MLB-StatsAPI
!pip install pyodbc

In [2]:
# Import packages
import statsapi as sa
import pandas as pd
from sqlalchemy import create_engine, DateTime, Integer, text
import urllib
import os
import pyodbc

# Create connection to SQL Server

In [3]:
from dotenv import load_dotenv

server = "localhost"
database = "MLBData"
username = "sa"
load_dotenv()
password = os.environ.get("SQLSAPassword")
driver = "ODBC Driver 17 for SQL Server"

# Build a clean connection string mapping
connection_string = f"DRIVER={{{driver}}};SERVER={server};DATABASE={database};UID={username};PWD={password};"

params = urllib.parse.quote_plus(connection_string)
connection_url = f"mssql+pyodbc:///?odbc_connect={params}"

# Inject security parameters
engine = create_engine(
    connection_url,
    connect_args={
        "Encrypt": "yes",
        "TrustServerCertificate": "yes"
    }
)

# Update players

In [6]:
# Fetch the 2026 player registry payload
resp = sa.get("sports_players", {"sportId": 1, "season": 2026})
players = pd.DataFrame(resp.get("people", []))

# Extract codes safely from the nested dictionaries
if "batSide" in players.columns:
    players["batSide"] = players["batSide"].apply(lambda x: x.get("code") if isinstance(x, dict) else None)
if "pitchHand" in players.columns:
    players["pitchHand"] = players["pitchHand"].apply(lambda x: x.get("code") if isinstance(x, dict) else None)

# Filter down to targeted analytical fields
target_cols = [
    "id", "firstName", "lastName", "primaryNumber", "birthDate",
    "height", "weight", "batSide", "pitchHand"
]
players = players[[col for col in target_cols if col in players.columns]]

# Cast data types to handle SQL constraints
players["birthDate"] = pd.to_datetime(players["birthDate"])
players["primaryNumber"] = players["primaryNumber"].astype("Int64")

# Insert directly into database
players.to_sql("players", con=engine, if_exists="replace", index=False)
print(f"Successfully processed and replaced {len(players)} rows in the players registry table.")

Successfully processed and replaced 1462 rows in the players registry table.


# Update Schedule and Teams

In [4]:
# Fetch the 2026 team registry payload
resp = sa.get("teams", {"sportId": 1, "season": 2026})
teams = pd.DataFrame(resp.get("teams"))

# Extract IDs from the nested league and division dictionaries
teams["leagueId"] = teams["league"].apply(lambda x: x.get("id") if isinstance(x, dict) else None)
teams["divisionId"] = teams["division"].apply(lambda x: x.get("id") if isinstance(x, dict) else None)
teams = teams[["id", "name", "teamCode", "locationName", "teamName", "leagueId", "divisionId"]]

# Insert directly into database
teams.to_sql("teams", con=engine, if_exists="replace", index=False)

30

In [5]:
# Fetch data from the API
resp = sa.get("schedule", {"sportId": 1, "season": 2026, "gameTypes": "R"})
games_list = [game for date in resp.get("dates", []) for game in date.get("games", [])]

# Extract embedded data during list construction to prevent SettingWithCopy warnings
for game in games_list:
    # Extract status code (e.g., "Final", "Live", "Preview")
    status = game.get("status", {}).get("abstractGameState")
    game["gameStatus"] = status

    # Extract home and away team IDs
    game["homeTeamId"] = game.get("teams", {}).get("home", {}).get("team", {}).get("id")
    game["awayTeamId"] = game.get("teams", {}).get("away", {}).get("team", {}).get("id")

    # Conditionally extract scores only if the game is Final
    if status == "Final":
        game["homeScoreFinal"] = game.get("teams", {}).get("home", {}).get("score")
        game["awayScoreFinal"] = game.get("teams", {}).get("away", {}).get("score")
    else:
        game["homeScoreFinal"] = None
        game["awayScoreFinal"] = None

# Create DataFrame and isolate required columns
sched = pd.DataFrame(games_list)

# 1. Parse as UTC, convert to Eastern Time (US/Eastern), and remove the timezone offset for SQL
sched["gameDate"] = pd.to_datetime(sched["gameDate"], utc=True).dt.tz_convert("US/Eastern").dt.tz_localize(None)

# 2. Extract the correct localized month and day
sched["gameMonth"] = sched["gameDate"].dt.month
sched["gameDay"] = sched["gameDate"].dt.day

target_columns = [
    "gamePk", "link", "gameType", "season", "gameDate", "gameMonth", "gameDay",
    "doubleHeader", "gameStatus", "homeTeamId", "awayTeamId", "homeScoreFinal", "awayScoreFinal"
]
schedCols = sched[target_columns].copy()

# Clean column types and enforce Nullable Integers for IDs and Scores
schedCols["season"] = schedCols["season"].astype("Int64")
schedCols["homeTeamId"] = schedCols["homeTeamId"].astype("Int64")
schedCols["awayTeamId"] = schedCols["awayTeamId"].astype("Int64")
schedCols["homeScoreFinal"] = schedCols["homeScoreFinal"].astype("Int64")
schedCols["awayScoreFinal"] = schedCols["awayScoreFinal"].astype("Int64")
schedCols["gameMonth"] = schedCols["gameMonth"].astype("Int64")
schedCols["gameDay"] = schedCols["gameDay"].astype("Int64")

# Safe insertion into SQL
schedCols.to_sql(
    "schedule",
    con=engine,
    if_exists="replace",
    index=False,
    dtype={"gameDate": DateTime()}
)

43

# Update PBP data

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import text, DateTime, Integer

def process_and_upsert_game(gamePK, sa_connection, db_engine):
    """
    Deletes existing rows for a given gamePK, parses play-by-play data,
    calculates dynamic team challenges remaining, injects pre-pitch context,
    and appends records to the DB.
    """
    gamePK = int(gamePK)

    #Create helper functions
    def safe_int(value):
        try: return int(value)
        except (TypeError, ValueError): return None

    def safe_datetime(value):
        try: return pd.to_datetime(value)
        except Exception: return pd.NaT

    def get_player_id(obj):
        if isinstance(obj, dict): return obj.get("id")
        return None

    # =========================================================================
    # NEW FEATURE: PITCH TYPE ID LOOKUP TABLE
    # =========================================================================
    PITCH_TYPE_LOOKUP = {
        "CH": 1,   # Changeup
        "CS": 2,   # Slow Curve
        "CU": 3,   # Curveball
        "EP": 4,   # Eephus
        "FA": 5,   # Fastball (unclassified)
        "FC": 6,   # Cutter
        "FF": 7,   # Four-Seam Fastball
        "FO": 8,   # Forkball
        "FS": 9,   # Splitter
        "KC": 10,  # Knuckle-Curve
        "KN": 11,  # Knuckleball
        "SI": 12,  # Sinker
        "SL": 13,  # Slider
        "ST": 14,  # Sweeper
        "SV": 15,  # Slurve
        "UN": 16   # Unknown Pitch Type
    }

    # Delete existing rows for this gamePk to avoid duplicates
    delete_statement = text("DELETE FROM pitch_by_pitch WHERE gamePk = :game_pk")
    with db_engine.begin() as conn:
        conn.execute(delete_statement, {"game_pk": gamePK})
        print(f"Cleared any existing rows for gamePk {gamePK} from the database.")

    # Fetch data via standard 'game' endpoint
    live_feed = sa_connection.get("game", {"gamePk": gamePK})
    if not live_feed:
        raise ValueError(f"No data returned for gamePk {gamePK}")

    # Grab nested objects from live_feed
    game_data = live_feed.get("gameData", {})
    home_team_id = game_data.get("teams", {}).get("home", {}).get("id")
    away_team_id = game_data.get("teams", {}).get("away", {}).get("id")
    live_data = live_feed.get("liveData", {})
    pbp = live_data.get("plays", {})
    boxscore = live_data.get("boxscore", {})
    all_plays = pbp.get("allPlays", [])

    # NEW FEATURE: Extract Home Plate Umpire ID
    hp_umpire_id = None
    boxscore_officials = live_data.get("boxscore", {}).get("officials", [])
    for official in boxscore_officials:
        if official.get("officialType") == "Home Plate":
            hp_umpire_id = official.get("official", {}).get("id")
            break

    # =========================================================================
    # STATE-MACHINE SETUP: PARSE UN-ALTERED STARTING DEFENSIVE LINEUPS
    # =========================================================================
    current_catchers = {"home": None, "away": None}
    for side in ["home", "away"]:
        team_box = boxscore.get("teams", {}).get(side, {})
        team_players = team_box.get("players", {})

        for p_id, p_info in team_players.items():
            bo = p_info.get("battingOrder")
            is_starter = bo is not None and int(bo) % 100 == 0
            all_positions = p_info.get("allPositions", [])
            if is_starter:
                if all_positions:
                    if str(all_positions[0].get("code")) == "2":
                        current_catchers[side] = p_info.get("person", {}).get("id")
                        break
                elif p_info.get("position", {}).get("code") == "2":
                    current_catchers[side] = p_info.get("person", {}).get("id")
                    break

        if not current_catchers[side]:
            for p_id, p_info in team_players.items():
                bo = p_info.get("battingOrder")
                is_starter = bo is not None and int(bo) % 100 == 0
                all_positions = p_info.get("allPositions", [])
                if is_starter and any(str(pos.get("code")) == "2" for pos in all_positions):
                    current_catchers[side] = p_info.get("person", {}).get("id")
                    break

    failed_home_challenges = 0
    failed_away_challenges = 0
    prev_inning = 1
    all_pitches = []

    # Process plays and events chronologically
    for play in all_plays:
        at_bat_idx = play.get("atBatIndex")
        about = play.get("about", {})
        inning = about.get("inning")

        if inning > prev_inning and inning > 9:
            if failed_home_challenges >= 2: failed_home_challenges = 1
            if failed_away_challenges >= 2: failed_away_challenges = 1
            prev_inning = inning

        half = about.get("halfInning")
        defending_side = "home" if half == "top" else "away"
        play_review = play.get("reviewDetails") if play.get("reviewDetails") else None
        play_desc = play.get("result", {}).get("description", "")
        matchup = play.get("matchup", {})
        batter_id = matchup.get("batter", {}).get("id")
        pitcher_id = matchup.get("pitcher", {}).get("id")

        is_abs_challenge_play = False
        if play_review and play_review.get("reviewType") == "MJ":
            is_abs_challenge_play = True

        # =========================================================================
        # FIX: DYNAMICALLY CALCULATE PRE-PLAY SCORES
        # Read the ending scores and subtract any runs scored during this plate appearance
        # =========================================================================
        play_result_context = play.get("result", {})
        post_play_home_score = play_result_context.get("homeScore", 0)
        post_play_away_score = play_result_context.get("awayScore", 0)

        # Count total runs scored by each team during this specific play array
        runners_list = play.get("runners", [])
        home_runs_scored_on_play = 0
        away_runs_scored_on_play = 0

        for r in runners_list:
            movement = r.get("movement", {})
            if movement.get("isOut") is False and movement.get("end") == "score":
                # Determine which team scored based on which team is batting
                if half == "top":
                    away_runs_scored_on_play += 1
                else:
                    home_runs_scored_on_play += 1

        pre_play_home_score = post_play_home_score - home_runs_scored_on_play
        pre_play_away_score = post_play_away_score - away_runs_scored_on_play

        # =========================================================================
        # FIX: EXTRACT PRE-PLAY BASERUNNERS
        # Read the 'runners' list. If a runner started on a base, record it.
        # =========================================================================
        runner_on_1st = 0
        runner_on_2nd = 0
        runner_on_3rd = 0

        for r in runners_list:
            start_base = r.get("movement", {}).get("start")
            if start_base == "1B": runner_on_1st = 1
            elif start_base == "2B": runner_on_2nd = 1
            elif start_base == "3B": runner_on_3rd = 1

        # Count Logic State Tracking (Display Count BEFORE pitch)
        current_balls = 0
        current_strikes = 0

        play_events = play.get("playEvents", [])
        for event in play_events:
            event_details = event.get("details", {})
            event_type = event_details.get("event", "")
            event_desc = event_details.get("description", "").lower()

            # Formally identify if this is a defensive roster shift
            is_sub = "substitution" in event_type.lower() or event.get("isSubstitution", False)

            # STRICT FIX: Look for directional text phrases ("to catcher") instead of checking generic descriptions
            is_switch = "switch" in event_type.lower() or "to catcher" in event_desc or "as catcher" in event_desc

            if is_sub or is_switch:
                sub_player_id = event.get("player", {}).get("id") or event_details.get("player", {}).get("id")
                position_obj = event.get("position", {}) or event_details.get("position", {})
                position_code = str(position_obj.get("code")) if isinstance(position_obj, dict) else ""

                # Check structural parameters first, then look for precise defensive entry syntax
                if position_code == "2" or "enters the game as catcher" in event_desc or "defensive switch to catcher" in event_desc:
                    if sub_player_id:
                        current_catchers[defending_side] = sub_player_id

            is_pitch = event.get("isPitch", False)
            event_review = event.get("reviewDetails") if event.get("reviewDetails") else None
            pitch_call_code = event_details.get("code", "")

            has_event_review = False
            if event_review and event_review.get("reviewType") == "MJ":
                has_event_review = True

            is_final_event = event == play_events[-1] if play_events else False
            has_play_level_challenge = is_final_event and is_abs_challenge_play
            is_any_abs = (has_event_review or has_play_level_challenge) and pitch_call_code != "F"

            if not is_pitch and not is_any_abs:
                continue

            catcher_id = current_catchers[defending_side]
            pitch_data_api = event.get("pitchData", {})

            px = pitch_data_api.get("coordinates", {}).get("pX")
            pz = pitch_data_api.get("coordinates", {}).get("pZ")
            sz_top = pitch_data_api.get("strikeZoneTop")
            sz_bot = pitch_data_api.get("strikeZoneBottom")

            active_review = None
            challenge_player_id = None
            challenge_position = None
            is_overturned = False

            if is_any_abs:
                active_review = event_review if event_review else play_review
                if active_review:
                    is_overturned = active_review.get("isOverturned", False)
                    if active_review.get("player"):
                        challenge_player_id = active_review.get("player", {}).get("id")
                        if challenge_player_id == batter_id: challenge_position = "Batter"
                        elif challenge_player_id == pitcher_id: challenge_position = "Pitcher"
                        elif challenge_player_id == catcher_id: challenge_position = "Catcher"
                        else: challenge_position = "Other"

            home_challenges_left = max(0, 2 - failed_home_challenges)
            away_challenges_left = max(0, 2 - failed_away_challenges)

            if is_any_abs and not is_overturned:
                if challenge_position == "Batter":
                    if half == "bottom": failed_home_challenges += 1
                    else: failed_away_challenges += 1
                elif challenge_position in ["Pitcher", "Catcher"]:
                    if half == "top": failed_home_challenges += 1
                    else: failed_away_challenges += 1

            # =========================================================================
            # FIX #2: Bulletproof Pre-Pitch Baserunner State from Event Count Object
            # =========================================================================
            # In the MLB API, the count block at the pitch level contains fields
            # like 'runnerOn1st', 'runnerOn2nd', 'runnerOn3rd'.
            # If they exist, they contain a dictionary with the player's info, else they are omitted.

            event_count_context = event.get("count", {})

            # Map the pitch type code string and extract its mapped lookup integer id
            pitch_type_code = event_details.get("type", {}).get("code")
            pitch_type_id = PITCH_TYPE_LOOKUP.get(pitch_type_code, PITCH_TYPE_LOOKUP["UN"])

            #runner_on_1st = 1 if event_count_context.get("runnerOn1st") else 0
            #runner_on_2nd = 1 if event_count_context.get("runnerOn2nd") else 0
            #runner_on_3rd = 1 if event_count_context.get("runnerOn3rd") else 0

            # Populate row using count tracking figures BEFORE the pitch applied
            pitch_data = {
                "gamePk": gamePK,
                "homeTeamId": home_team_id,
                "awayTeamId": away_team_id,
                "atBatIndex": at_bat_idx,
                "pitchNumber": event.get("pitchNumber"),
                "playId": event.get("playId"),
                "inning": inning,
                "inningHalf": 0 if half == "top" else 1 if half == "bottom" else None,
                "batterId": batter_id,
                "pitcherId": pitcher_id,
                "catcherId": catcher_id,
                "homePlateUmpireId": hp_umpire_id,          # INJECTED FIELD
                "prePitchBalls": current_balls,             # UPDATED LOGIC (Before pitch context)
                "prePitchStrikes": current_strikes,         # UPDATED LOGIC (Before pitch context)
                "prePlayHomeScore": pre_play_home_score,    # INJECTED FIELD
                "prePlayAwayScore": pre_play_away_score,    # INJECTED FIELD
                "runnerOn1st": runner_on_1st,               # INJECTED FIELD
                "runnerOn2nd": runner_on_2nd,               # INJECTED FIELD
                "runnerOn3rd": runner_on_3rd,               # INJECTED FIELD
                "outs": event.get("count", {}).get("outs"), # Keep outs as recorded
                "pitchCall": pitch_call_code,
                "pitchDescription": event_details.get("description") or play_desc,
                "gameDate": safe_datetime(event.get("startTime")),
                "pitchType": event_details.get("type", {}).get("code"),
                "pitchTypeId": pitch_type_id,
                "startSpeed": pitch_data_api.get("startSpeed"),
                "plateTimeX": px,
                "plateTimeY": pz,
                "szTop": sz_top,
                "szBottom": sz_bot,
                "szTopRule": sz_top,
                "szBottomRule": sz_bot,
                "homeChallengesLeft": home_challenges_left,
                "awayChallengesLeft": away_challenges_left,
                "hasABSChallenge": int(is_any_abs),
                "challengeReason": active_review.get("reason") if is_any_abs and active_review and active_review.get("reason") else None,
                "challengeResult": "Overturned" if is_any_abs and is_overturned else "Confirmed" if is_any_abs else None,
                "isOverturned": is_overturned if is_any_abs else False,
                "challengePlayerId": challenge_player_id,
                "challengePlayerPosition": challenge_position,
            }
            all_pitches.append(pitch_data)

            # UPDATE THE STATE LOGIC VARIABLES FOR THE NEXT PITCH IN SEQUENCE
            # Extract post-event API count so next loops process accurate numbers
            post_event_count = event.get("count", {})
            current_balls = post_event_count.get("balls", current_balls)
            current_strikes = post_event_count.get("strikes", current_strikes)

    if not all_pitches:
        print(f"No pitches found for gamePk {gamePK}. Skipping database write.")
        return

    pitches_df = pd.DataFrame(all_pitches)

    # Spatial metrics
    ball_radius = 0.1208
    plate_half_width = 0.7083
    limit_x = plate_half_width + ball_radius

    def compute_spatial_metrics(row):
        x = row["plateTimeX"]
        y = row["plateTimeY"]
        t = row["szTop"]
        b = row["szBottom"]

        if pd.isna(x) or pd.isna(y) or pd.isna(t) or pd.isna(b):
            return "Unknown", np.nan

        x = float(x)
        y = float(y)
        t = float(t)
        b = float(b)

        is_strike_x = abs(x) <= limit_x
        is_strike_y = (y >= (b - ball_radius)) and (y <= (t + ball_radius))
        zone_string = "Strike" if (is_strike_x and is_strike_y) else "Ball"

        dx = max(0.0, abs(x) - limit_x)
        dy = 0.0
        if y > (t + ball_radius):
            dy = y - (t + ball_radius)
        elif y < (b - ball_radius):
            dy = (b - ball_radius) - y

        miss_dist = np.sqrt(dx**2 + dy**2) if (dx > 0 or dy > 0) else 0.0
        return zone_string, miss_dist

    metrics = pitches_df.apply(compute_spatial_metrics, axis=1)
    pitches_df["calculatedZoneResult"] = [m[0] for m in metrics]
    pitches_df["distanceFromZoneEdge"] = [m[1] for m in metrics]

    int_cols = [
        "gamePk", "homeTeamId", "awayTeamId", "atBatIndex", "pitchNumber",
        "inning","prePitchBalls", "prePitchStrikes", "prePlayHomeScore",
        "prePlayAwayScore","runnerOn1st", "runnerOn2nd", "runnerOn3rd",
        "outs", "batterId", "pitcherId","catcherId", "homePlateUmpireId",
        "homeChallengesLeft", "awayChallengesLeft","hasABSChallenge", "challengePlayerId"
    ]
    for col in int_cols:
        if col in pitches_df.columns:
            pitches_df[col] = pd.to_numeric(pitches_df[col], errors="coerce").astype("Int64")

    float_cols = [
        "startSpeed", "plateTimeX", "plateTimeY", "szTop", "szBottom",
        "szTopRule", "szBottomRule", "distanceFromZoneEdge"
    ]
    for col in float_cols:
        if col in pitches_df.columns:
            pitches_df[col] = pd.to_numeric(pitches_df[col], errors="coerce").astype("float64")

    pitches_df["gameDate"] = pd.to_datetime(pitches_df["gameDate"], errors="coerce")

    pitches_df.to_sql(
        "pitch_by_pitch",
        con=db_engine,
        if_exists="append",
        index=False,
        chunksize=500,
        dtype={
            "gameDate": DateTime(),
            "gamePk": Integer(),
        },
    )

    print(f"Successfully processed and appended {len(pitches_df)} rows for gamePk {gamePK}.")

In [ ]:
# To run PBP import for single game, just uncomment and update GamePK in line below
# process_and_upsert_game(824513, sa, engine)

In [ ]:
# Run PBP Import for up to 50 new games
select_statement = text("""
    SELECT DISTINCT TOP 50 GamePK
    FROM MLBData..Schedule
    WHERE gameStatus = 'Final'
      AND GamePK NOT IN (
          SELECT DISTINCT GamePK
          FROM MLBData..pitch_by_pitch
      )
""")

with engine.connect() as conn:
    gamePKs = conn.execute(select_statement).fetchall()

for (gamePK,) in gamePKs:
    try:
        process_and_upsert_game(gamePK, sa, engine)
    except Exception as e:
        print(f"Failed for gamePk {gamePK}: {e}")